In [46]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [47]:
file_path = r"D:\clv\online_retail_II.xlsx"

all_sheets = pd.read_excel(file_path, sheet_name=None)

df = pd.concat(all_sheets.values(), ignore_index=True)

print(df.shape)

df.head()

(1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [48]:
clean_df = df.copy()

In [49]:
clean_df.columns = (
    clean_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

clean_df.columns

Index(['invoice', 'stockcode', 'description', 'quantity', 'invoicedate',
       'price', 'customer_id', 'country'],
      dtype='str')

In [50]:
clean_df["invoice"] = clean_df["invoice"].astype(str)

clean_df["stockcode"] = clean_df["stockcode"].astype(str)

clean_df["invoicedate"] = pd.to_datetime(clean_df["invoicedate"])

clean_df["customer_id"] = pd.to_numeric(
    clean_df["customer_id"],
    errors="coerce"
)

In [51]:
print("Before:", clean_df.shape)

clean_df = clean_df.drop_duplicates().copy()

print("After:", clean_df.shape)

Before: (1067371, 8)
After: (1033036, 8)


In [52]:
print(clean_df["customer_id"].isna().sum())

clean_df = clean_df.dropna(
    subset=["customer_id"]
).copy()

clean_df["customer_id"] = clean_df["customer_id"].astype(int)

print(clean_df["customer_id"].isna().sum())

235151
0


In [55]:
clean_df["description"] = clean_df["description"].fillna(
    "Unknown Product"
)

In [56]:
for col in clean_df.select_dtypes(include="object"):

    clean_df[col] = clean_df[col].str.strip()

C:\Users\Urvi Patel\AppData\Local\Temp\ipykernel_20568\2129197972.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in clean_df.select_dtypes(include="object"):


In [57]:
clean_df["country"] = clean_df["country"].str.title()

clean_df["description"] = clean_df["description"].str.upper()

In [58]:
clean_df["is_cancelled"] = (
    clean_df["invoice"]
    .str.startswith("C")
)

clean_df["is_return"] = (
    clean_df["quantity"] < 0
)

In [59]:
analysis_df = clean_df.copy()

analysis_df = analysis_df[
    (~analysis_df["is_cancelled"])
    &
    (~analysis_df["is_return"])
].copy()

In [60]:
analysis_df = analysis_df[
    analysis_df["price"] > 0
].copy()

In [61]:
analysis_df = analysis_df[
    analysis_df["quantity"] > 0
].copy()

In [62]:
analysis_df["revenue"] = (
    analysis_df["quantity"]
    *
    analysis_df["price"]
)

In [63]:
analysis_df["year"] = analysis_df["invoicedate"].dt.year

analysis_df["month"] = analysis_df["invoicedate"].dt.month

analysis_df["month_name"] = (
    analysis_df["invoicedate"]
    .dt.month_name()
)

analysis_df["day"] = analysis_df["invoicedate"].dt.day

analysis_df["day_name"] = (
    analysis_df["invoicedate"]
    .dt.day_name()
)

analysis_df["hour"] = (
    analysis_df["invoicedate"]
    .dt.hour
)

analysis_df["quarter"] = (
    analysis_df["invoicedate"]
    .dt.quarter
)

analysis_df["week"] = (
    analysis_df["invoicedate"]
    .dt.isocalendar()
    .week
    .astype(int)
)

In [64]:
print("Shape:", analysis_df.shape)

print()

print("Missing Values")
print(analysis_df.isna().sum())

print()

print("Duplicate Rows:",
      analysis_df.duplicated().sum())

print()

print("Customer IDs:",
      analysis_df["customer_id"].isna().sum())

print()

print("Invoice Missing:",
      analysis_df["invoice"].isna().sum())

print()

print("StockCode Missing:",
      analysis_df["stockcode"].isna().sum())

Shape: (779425, 19)

Missing Values
invoice         0
stockcode       0
description     0
quantity        0
invoicedate     0
price           0
customer_id     0
country         0
is_cancelled    0
is_return       0
revenue         0
year            0
month           0
month_name      0
day             0
day_name        0
hour            0
quarter         0
week            0
dtype: int64

Duplicate Rows: 0

Customer IDs: 0

Invoice Missing: 0

StockCode Missing: 0


In [65]:
clean_df.to_csv(
    "clean_with_returns.csv",
    index=False
)

analysis_df.to_csv(
    "clean_online_retail.csv",
    index=False
)

print("Datasets saved successfully.")

Datasets saved successfully.
